In [14]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Sequence
import numpy as np

In [15]:

ATOMIC_NUMBER = {"H": 1, "C": 6, "N":7, "O":8, "B":5}
ELEMENT_SYMBOL = {v:k for k,v in ATOMIC_NUMBER.items()} # dict comprehension, erstellt ein neues dict, indem die keys und values vertauscht werden

@dataclass     # iinit funktion wird damit schon definiert
class Atom:
    symbol: str
    coord: Sequence[float]

    def __post_init__(self) -> None:   #liefert nichts zurück
        self.symbol = self.symbol.strip().capitalize()
        self.coord = np.asarray(self.coord, dtype = float)

    def __str__(self):   #helferfuntionenn dunder-Funktion __name__
        return f"{self.symbol} {self.coord[0]:.6f} {self.coord[1]:.6f} {self.coord[2]:.6f}"
     
    @property
    def atomic_number(self) -> int:
        return ATOMIC_NUMBER[self.symbol]
    
    @property
    def n_electrons(self) -> int:
        return self.atomic_number
    


    @classmethod
    def from_string(cls, s: str) -> Atom:
        parts = s.split()   # spaltet string in liste von substrings
        if len(parts) != 4:
            raise ValueError(f"Expected 'Symbol X Y Z', got {s}")
        symbol = parts[0]   # 0. element der liste
        try:
            coord = np.asarray(parts[1:], dtype=float)
        except ValueError as exc:
            raise ValueError(f"Invalid coordinates in atom string {s}") from exc
        return cls(symbol, coord)

    def get_distance(self, other: Atom) -> float:
        if not isinstance(other, Atom): 
            raise ValueError(f"Distance can only be calculated between two Atom intstances")
        return float(np.linalg.norm(self.coord - other.coord))


In [16]:
hatom1 = Atom('h', [0.0, 1.0, 2.0])
hatom2 = Atom('H', [0.0, 0.0, 2.0])
print(hatom1)
print(hatom2)

H 0.000000 1.000000 2.000000
H 0.000000 0.000000 2.000000


In [17]:
s = "C 0.0 0.0 0.0"
c_atom = Atom.from_string(s)
h_atom = Atom("h", np.zeros(3))
h_atom.__str__()


'H 0.000000 0.000000 0.000000'

In [18]:
atom1 = Atom("C", np.zeros(3))
atom2 = Atom("C", np.ones(3))
atom1.get_distance(atom2)
print(atom2.atomic_number)
print(atom2.n_electrons)

6
6


In [19]:
@dataclass
class Molecule:
    atoms: List[Atom]

    @classmethod
    def from_string(cls, xyz_string: str) -> Molecule:
        lines = xyz_string.strip().splitlines()
        atoms = []
        for line in lines[2:]:
            atom = Atom.from_string(line)
            atoms.append(atom)
        return cls(atoms)
    
    def __str__(self):
        return "\n".join([str(atom) for atom in self.atoms])   # string abc das jeden wert in neue zeile schreibt

    @property
    def n_electrons(self):
       return sum([atom.n_electrons for atom in self.atoms])   # list comprehension
    
    def to_string(self) -> str:   # hier wird die xyz string generiert, die dann für die visualisierung genutzt werden kann
        lines = [str(len(self.atoms)), "Generated by Molecule Class"] 
        for atom in self.atoms:
            line = f"{atom.symbol} {atom.coord[0]:.6f} {atom.coord[1]:.6f} {atom.coord[2]:.6f}"
            lines.append(line)
        return "\n".join(lines) # string abc das jeden wert in neue zeile schreibt  

        
 

In [20]:
atomlist = [atom1, atom2]
mol = Molecule(atomlist)

In [21]:
benzene_xyz = """12
Benzene molecule
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000
"""
benzene = Molecule.from_string(benzene_xyz)
print(benzene)
print(benzene.to_string())

C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000
12
Generated by Molecule Class
C 0.000000 1.402720 0.000000
C 1.214790 0.701360 0.000000
C 1.214790 -0.701360 0.000000
C 0.000000 -1.402720 0.000000
C -1.214790 -0.701360 0.000000
C -1.214790 0.701360 0.000000
H 0.000000 2.490290 0.000000
H 2.156660 1.245150 0.000000
H 2.156660 -1.245150 0.000000
H 0.000000 -2.490290 0.000000
H -2.156660 -1.245150 0.000000
H -2.156660 1.245150 0.000000


In [22]:
import py3Dmol

def visualize(molecule: Molecule):
    xyz = molecule.to_string()  
    view = py3Dmol.view(width=400, height=400)
    view.addModel(xyz, "xyz") # hier muss noch die xyz string übergeben werden, die aus molecule generiert werden muss
    view.setStyle({'stick': {}})
    view.zoomTo()
    return view


In [23]:
view = visualize(benzene)
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [24]:
import numpy as np
from dataclasses import field

ODD_DF = np.array([1, 1, 2, 3, 8, 15, 48, 105, 384, 945])
PI = np.pi

def l_to_ijk(L):
    IJK = []
    for I in range(L, -1 , -1):
        for J in range(L - I, -1, -1):
            IJK.append((I, J, L - I - J))
    return sorted(IJK, reverse = True)

def primitive_cart_norm(alpha, l, m, n):
    L = l + m + n
    num = (2 * alpha/PI)**(3/4) * (4 * alpha)**(L/2)
    den = np.sqrt(ODD_DF[l] * ODD_DF[m] * ODD_DF[n]) # hier wird die normierungsfaktor berechnet, der dafür sorgt, dass die gaussfunktion normiert ist, also dass das integral über alle raum gleich 1 ist
    return num/den

@dataclass
class Shell:
    l : int # 0 s, 1 p, 2 d, 3 f 
    exponents: np.ndarray # 1d array of exponents
    coefficients: np.ndarray # 1d array of coefficients
    norm_factors : np.ndarray = field(init=False) # wird nicht im init constructor übergeben, sondern wird automatisch berechnet
    #center: np.ndarray = field(init=False)

    def __post_init__(self):
        self.norm_factors = self.get_norm_factors()

    def get_norm_factors(self):
        ijk = l_to_ijk(self.l)
        norm_factors = np.empty((len(self.exponents), len(ijk)), dtype = np.float64)
        for i, (l, m, n) in enumerate(ijk):
            norm_factors[:,i] = primitive_cart_norm(self.exponents, l, m, n)
        return norm_factors



In [25]:
import requests
import json


@dataclass  # basissatz aus dictonary aus shells
class BasisSet:
    name: str
    elements: dict[str, list[Shell]] = field(default_factory=dict) # default_factory ist eine Funktion, die aufgerufen wird, um den Standardwert für das Feld zu erstellen. In diesem Fall wird eine leere dict erstellt, wenn kein Wert übergeben wird.
    
    def download_from_bse(self, element_list: list, timeout: int = 30) -> None:
        url = f"https://www.basissetexchange.org/api/basis/{self.name}/format/json/"
        url += f"?elements={','.join(element_list)}"
        response = requests.get(url, timeout=timeout)
        data_json = response.json()
        print(f"Dump data to {self.name}.json")
        json.dump(data_json, open(f"{self.name}.json", "w"), indent = 4)
        self.parse_elements(data_json)

    def parse_elements(self, data: dict) -> None:
        self.elements = {} # leeres dict, das mit den neuen Daten gefüllt wird
        basis_data = data.get("elements", {}) # get methode von dict, die den wert für den key "elements" zurückgibt, oder ein leeres dict, wenn der key nicht existiert
        for atomic_number, element_data in basis_data.items():
            symbol = ELEMENT_SYMBOL.get(int(atomic_number))
            shells: list[Shell] = []
            for shell_data in element_data.get("electron_shells", []):
                angular_momentum = shell_data["angular_momentum"]
                contraction_coefficients = shell_data["coefficients"]
                exponents = shell_data["exponents"]
                for l, coefficients in zip(angular_momentum, contraction_coefficients):
                    shells.append(Shell(l = int(l),
                                        exponents = np.asarray(exponents, dtype=float),
                                        coefficients = np.asarray(coefficients, dtype=float)))
                self.elements[symbol] = shells

    def __str__(self) -> str:
        output = f"Basis set: {self.name}\n"
        for symbol, shells in self.elements.items():
            output += f"Element: {symbol}\n"
            for shell in shells:
                output += f"l = {shell.l}, exponents = {shell.exponents}, coefficients = {shell.coefficients}\n"
                output += f"norm_factors = {shell.norm_factors}"
        return output
                

In [26]:
sto3g = BasisSet("sto-3g")
sto3g.download_from_bse(["H", "C", "O", "N", "B"])
print(sto3g)

Dump data to sto-3g.json
Basis set: sto-3g
Element: H
l = 0, exponents = [3.42525091 0.62391373 0.1688554 ], coefficients = [0.15432897 0.53532814 0.44463454]
norm_factors = [[1.79444183]
 [0.50032649]
 [0.18773546]]Element: B
l = 0, exponents = [48.79111318  8.88736217  2.40526704], coefficients = [0.15432897 0.53532814 0.44463454]
norm_factors = [[13.15726547]
 [ 3.66851037]
 [ 1.37652013]]l = 0, exponents = [2.23695614 0.5198205  0.16906176], coefficients = [-0.09996723  0.39951283  0.70011547]
norm_factors = [[1.30362654]
 [0.436315  ]
 [0.18790751]]l = 1, exponents = [2.23695614 0.5198205  0.16906176], coefficients = [0.15591627 0.60768372 0.39195739]
norm_factors = [[3.89952694 3.89952694 3.89952694]
 [0.62915382 0.62915382 0.62915382]
 [0.15452431 0.15452431 0.15452431]]Element: C
l = 0, exponents = [71.61683735 13.04509632  3.53051216], coefficients = [0.15432897 0.53532814 0.44463454]
norm_factors = [[17.54573016]
 [ 4.89210263]
 [ 1.83564365]]l = 0, exponents = [2.94124936 0.